# Week 2 — Advanced Feature Engineering
### Restaurant Demand Forecasting / Inventory Optimization

This notebook creates the time-series features that give our forecasting model predictive power. We build date components, holiday indicators, lag features, rolling statistics, and a clean sequential train/validation/test split.

## 1. Setup and Data Loading

Place the required CSV files inside `data/raw/` or keep them in the original `Food & Restaurant Services` folder if you have not moved the dataset yet. The notebook will automatically detect the expected path.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

project_root = Path.cwd()
raw_candidates = [
    project_root / 'data' / 'raw',
    project_root / 'data',
    Path('/Users/user/Documents/Food & Restaurant Services'),
    Path('C:/Users/user/Documents/Food & Restaurant Services')
]

raw_dir = None
for candidate in raw_candidates:
    if candidate.exists():
        raw_dir = candidate
        break

if raw_dir is None:
    raise FileNotFoundError(
        'Could not find the raw dataset folder.
'
        'Place train.csv, test.csv, stores.csv, oil.csv, holidays_events.csv, transactions.csv, and sample_submission.csv inside data/raw/'
    )

print('Using raw data from:', raw_dir)

paths = {
    'train' : raw_dir / 'train.csv',
    'test'  : raw_dir / 'test.csv',
    'stores': raw_dir / 'stores.csv',
    'oil'   : raw_dir / 'oil.csv',
    'holidays': raw_dir / 'holidays_events.csv',
    'transactions': raw_dir / 'transactions.csv',
    'sample_submission': raw_dir / 'sample_submission.csv',
}
for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')

train_df = pd.read_csv(paths['train'])
test_df = pd.read_csv(paths['test'])
stores_df = pd.read_csv(paths['stores'])
oil_df = pd.read_csv(paths['oil'])
hol_df = pd.read_csv(paths['holidays'])
txn_df = pd.read_csv(paths['transactions'])
sample_submission = pd.read_csv(paths['sample_submission'])

for df, col in [(train_df, 'date'), (test_df, 'date'), (oil_df, 'date'), (hol_df, 'date'), (txn_df, 'date')]:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print('Loaded datasets:')
print(' ', 'train:', train_df.shape)
print(' ', 'test :', test_df.shape)
print(' ', 'stores:', stores_df.shape)
print(' ', 'oil  :', oil_df.shape)
print(' ', 'holidays:', hol_df.shape)
print(' ', 'transactions:', txn_df.shape)

## 2. Create a Daily Forecasting Dataset

We aggregate sales into daily demand totals and merge external signals like oil price, transactions, store metadata, and holidays.

In [ ]:
daily_sales = (
    train_df.groupby('date')['sales']
    .sum()
    .reset_index()
    .rename(columns={'sales': 'total_sales'})
)

daily_promo = (
    train_df.groupby('date')['onpromotion']
    .sum()
    .reset_index()
    .rename(columns={'onpromotion': 'total_promo_items'})
)

daily_txn = (
    txn_df.groupby('date')['transactions']
    .sum()
    .reset_index()
)

daily = (
    daily_sales
    .merge(daily_promo, on='date', how='left')
    .merge(daily_txn, on='date', how='left')
    .merge(oil_df.rename(columns={'dcoilwtico': 'oil_price'}), on='date', how='left')
)

daily = daily.sort_values('date').reset_index(drop=True)

# Fill continuous daily dates and interpolate support columns
calendar = pd.DataFrame({'date': pd.date_range(daily['date'].min(), daily['date'].max(), freq='D')})
daily = calendar.merge(daily, on='date', how='left')
daily['total_sales'] = daily['total_sales'].fillna(0)
daily['total_promo_items'] = daily['total_promo_items'].fillna(0)
daily['transactions'] = daily['transactions'].interpolate(method='linear')
daily['oil_price'] = daily['oil_price'].interpolate(method='linear')

daily['is_missing_sales'] = daily['total_sales'].isna().astype(int)
daily['missing_sales'] = daily['total_sales'].isnull().sum()

print('Daily aggregation complete: rows =', len(daily))
daily.head()

## 3. Date Features and Holiday Indicators

Create the core time features that models rely on, and flag holidays using the original holiday event file.

In [ ]:
daily['year'] = daily['date'].dt.year
daily['month'] = daily['date'].dt.month
daily['day'] = daily['date'].dt.day
daily['dayofweek'] = daily['date'].dt.dayofweek
daily['weekofyear'] = daily['date'].dt.isocalendar().week.astype(int)
daily['quarter'] = daily['date'].dt.quarter
daily['is_weekend'] = (daily['dayofweek'] >= 5).astype(int)
daily['is_month_start'] = daily['date'].dt.is_month_start.astype(int)
daily['is_month_end'] = daily['date'].dt.is_month_end.astype(int)

holiday_flags = (
    hol_df[['date', 'type', 'locale', 'description']]
    .copy()
    .assign(is_holiday=1)
)

holiday_flags['holiday_level'] = holiday_flags['locale'].map({
    'National': 3,
    'Regional': 2,
    'Local': 1,
}).fillna(0).astype(int)

daily = daily.merge(holiday_flags[['date', 'is_holiday', 'holiday_level']], on='date', how='left')
daily['is_holiday'] = daily['is_holiday'].fillna(0).astype(int)
daily['holiday_level'] = daily['holiday_level'].fillna(0).astype(int)

print('Holiday flags created:', daily['is_holiday'].sum(), 'holiday rows')
daily[['date','is_holiday','holiday_level']].head(8)

## 4. Lag Features and Rolling Statistics

Generate features that capture recent demand history without leaking future information.

In [ ]:
lags = [7, 14, 28]
for lag in lags:
    daily[f'sales_lag_{lag}'] = daily['total_sales'].shift(lag)

daily['rolling_mean_7'] = daily['total_sales'].shift(1).rolling(7, min_periods=1).mean()
daily['rolling_std_14'] = daily['total_sales'].shift(1).rolling(14, min_periods=1).std().fillna(0)
daily['rolling_max_7'] = daily['total_sales'].shift(1).rolling(7, min_periods=1).max()
daily['promo_lag_7'] = daily['total_promo_items'].shift(7)
daily['promo_rolling_7'] = daily['total_promo_items'].shift(1).rolling(7, min_periods=1).mean()

daily['sales_log1p'] = np.log1p(daily['total_sales'])

feature_columns = [
    'date', 'total_sales', 'sales_log1p', 'total_promo_items', 'transactions', 'oil_price',
    'year', 'month', 'day', 'dayofweek', 'weekofyear', 'quarter', 'is_weekend',
    'is_month_start', 'is_month_end', 'is_holiday', 'holiday_level',
    'sales_lag_7', 'sales_lag_14', 'sales_lag_28',
    'rolling_mean_7', 'rolling_std_14', 'rolling_max_7',
    'promo_lag_7', 'promo_rolling_7'
]

daily_features = daily[feature_columns].copy()
daily_features = daily_features.dropna().reset_index(drop=True)

print('Feature dataset shape after lag feature dropna:', daily_features.shape)
daily_features.head()

## 5. Sequential Split: Train / Validation / Test

We keep the split sequential to prevent look-ahead leakage. The test set is the final 2 months of the series.

In [ ]:
daily_features['date'] = pd.to_datetime(daily_features['date'])
last_day = daily_features['date'].max()
test_start = last_day - pd.DateOffset(months=2) + pd.Timedelta(days=1)
val_start = test_start - pd.DateOffset(months=2)

train_set = daily_features[daily_features['date'] < val_start].copy()
val_set = daily_features[(daily_features['date'] >= val_start) & (daily_features['date'] < test_start)].copy()
test_set = daily_features[daily_features['date'] >= test_start].copy()

print('Train rows:', len(train_set))
print('Validation rows:', len(val_set))
print('Test rows:', len(test_set))

for name, df in [('Train', train_set), ('Validation', val_set), ('Test', test_set)]:
    print(f'{name}: {df[date].min().date()} → {df[date].max().date()} | rows = {len(df)}')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train_set['date'], train_set['total_sales'], label='Train', alpha=0.65)
ax.plot(val_set['date'], val_set['total_sales'], label='Validation', alpha=0.85)
ax.plot(test_set['date'], test_set['total_sales'], label='Test', alpha=0.85)
ax.legend()
ax.set_title('Sequential Train / Validation / Test Split')
ax.set_xlabel('Date')
ax.set_ylabel('Total Sales')
plt.tight_layout()
plt.show()

## 6. Save Processed Feature Dataset

Save the engineered dataset for Week 3 model training.

In [ ]:
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
daily_features.to_csv(processed_dir / 'week2_features.csv', index=False)
train_set.to_csv(processed_dir / 'train_features.csv', index=False)
val_set.to_csv(processed_dir / 'val_features.csv', index=False)
test_set.to_csv(processed_dir / 'test_features.csv', index=False)

print('Saved feature datasets to:', processed_dir)
print('  week2_features.csv       ', daily_features.shape)
print('  train_features.csv       ', train_set.shape)
print('  val_features.csv         ', val_set.shape)
print('  test_features.csv        ', test_set.shape)